# **Preparo do ambiente**

### Montando Drive

In [1]:
# Montando Drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/TCC/

Mounted at /content/drive/
/content/drive/MyDrive/TCC


In [1]:
!ls

'Ações do GitHub.ipynb'		 Models		    imgs
'Análise dos Resultados.ipynb'	 OpenGraph.ipynb    link_prediction
'Análise dos Resultados.ipynb'	 README.md	    node_classification
 History			 datasets	    requirements.txt
 LICENSE			 graph_generation   resultados_df.csv


In [2]:
%cd OpenGraph


/content/drive/MyDrive/TCC/OpenGraph


### Baixando dependências

In [13]:
# Instalando bibliotecas
!pip install torch_geometric
!pip install setproctitle

In [2]:
# Imports
import os
import shutil
import subprocess
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.sparse import coo_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, auc, mean_squared_error
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from scipy.sparse.csgraph import connected_components
from scipy.sparse import lil_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score

/dados/luiz.gontijo/TCC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Definindo funções

In [3]:
def calcula_metricas(y_test, y_pred):
    qnt_zeroes = (y_pred == 0).sum()
    qnt_ones = (y_pred == 1).sum()
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    return qnt_zeroes, qnt_ones, accuracy, precision, recall, f1

In [4]:
def print_metricas(nome_classificador, qnt_zeroes, qnt_ones, accuracy, precision, recall, f1):
    print(f'{nome_classificador}:')
    print(f'Elementos 0: {qnt_zeroes}')
    print(f'Elementos 1: {qnt_ones}')
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1-score: {f1}')

In [5]:
def bizoia_resultados(nome_classificador: str, n: int, nome_dataset: str, adj_file: str, mask_file: str, embedding_size: int):
  # bizoia n resultados e dá a média dos trem
  media = {'accuracy': 0, 'precision': 0, 'recall': 0}
  for i in range(n):
      print(f'--- Resultado {i} ---')
      with open('node_classification/Resultados/predicoes/'+nome_dataset+'/adj'+('_' if adj_file else '')+adj_file+'_mask'+('_' if mask_file else '')+mask_file +'/predict'+str(i)+'.pkl', 'rb') as f:
          data = pickle.load(f)

      preds = data['preds']
      labels = data['labels']
      nodes = data['nodes']

      resultado = pd.DataFrame.from_dict({'preds': data['preds'], 'labels': data['labels']})

      with open('node_classification/Resultados/embeddings/'+nome_dataset+'/adj'+('_' if adj_file else '')+adj_file+'_mask'+('_' if mask_file else '')+mask_file +'/embedding'+str(i)+'.pkl', 'rb') as f:
          embed = pickle.load(f)

      s = 0
      for e in embed:
          s += len(e)

      print('Nós embeddados:', s)
      print('Nós labelados:', len(labels))
      print('Porcentagem de nós embeddados:', s/embedding_size, end='\n\n')

      print(resultado['preds'].value_counts(), end='\n\n')
      print(resultado['labels'].value_counts(), end='\n\n')

      qnt_zeroes, qnt_ones, accuracy, precision, recall, f1 = calcula_metricas(resultado['labels'], resultado['preds'])
      media['accuracy'] += accuracy
      media['precision'] += precision
      media['recall'] += recall

      print_metricas(f'{nome_classificador}: iteração {i}', qnt_zeroes, qnt_ones, accuracy, precision, recall, f1)
      print()
      print()

  print('--- Média dos resultados ---')
  print('Accuracy:', media['accuracy']/n)
  print('Precision:', media['precision']/n)
  print('Recall:', media['recall']/n)
  print()


In [6]:
def remove_onlysenders(dataset: pd.DataFrame):
  '''
    Somente aplicável em datasets provenientes do SAML-D. Reduz iterativamente o dataset removendo arestas cujo sender nunca foi receiver, até todos terem sido receivers.
  '''
  df = dataset
  iteration = 0
  while True:
    accounts = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
    accounts_set = set(accounts)

    senders = set(df['Sender_account'])
    receivers = set(df['Receiver_account'])

    #print(f'Quantos accounts pssuem arestas bidirecionais?   {len(senders.intersection(receivers))} ({len(senders.intersection(receivers))/len(accounts_set)*100}%)')
    #print(f'Quantos accounts foram só senders?              {len(senders - receivers)} ({len(senders - receivers)/len(accounts_set)*100}%)')
    #print(f'Quantos accounts foram só receivers?            {len(receivers - senders)} ({len(receivers - senders)/len(accounts_set)*100}%)')

    bad_transactions = df[df['Sender_account'].isin(senders - receivers) == True]

    #print('Iteração', iteration)
    #print(f'Total de transações no dataset:   {df.size}')
    #print(f'Quantidade de "transações ruins": {bad_transactions.size} ({bad_transactions.size/df.size * 100})')

    if(bad_transactions.size == 0):
      very_good_transactions = df
      break
    else:
      iteration += 1

    # Quantos receiver accounts são só receivers
    #df[df['Receiver_account'].isin(receivers - senders) == True].size/df.size * 100
    #print('Quantidade de receiver que são só receivers:', df[df['Receiver_account'].isin(receivers - senders) == True].size/df.size * 100)

    good_transactions = df[df['Sender_account'].isin(senders - receivers) == False]

    df = good_transactions
    #print()
    #print('='*50)
    #print()

  print('Última iteração:', iteration)
  print('Quantidade de nós restantes:', very_good_transactions['Receiver_account'].unique().size)
  print('Porcentagem de transações removidas:', very_good_transactions.size/dataset.size)
  return very_good_transactions

# **Leitura dos Datasets**

#### Carregamento dos tipos de lavagem
Tipos de lavagem carregados por fora porque alguns podem estar ausentes em alguns dos recortes do SAML-D

In [ ]:
laundering_types = [
    # --- Tipos Normais (Simulações de comportamento lícito) ---
    'Normal_Cash_Deposits',
    'Normal_Cash_Withdrawal',
    'Normal_Fan_In',
    'Normal_Fan_Out',
    'Normal_Foward',
    'Normal_Group',
    'Normal_Mutual',
    'Normal_Periodical',
    'Normal_Plus_Mutual',
    'Normal_Small_Fan_Out',
    'Normal_single_large',

    # --- Tipos de Lavagem (Tipologias e Técnicas Suspeitas) ---
    'Behavioural_Change_1',
    'Behavioural_Change_2',
    'Bipartite',
    'Cash_Withdrawal',  # Classificado como anômalo neste dataset
    'Cycle',
    'Deposit-Send',
    'Fan_In',           # Padrão de agrupamento suspeito
    'Fan_Out',          # Padrão de dispersão suspeito
    'Gather-Scatter',
    'Layered_Fan_In',
    'Layered_Fan_Out',
    'Over-Invoicing',
    'Scatter-Gather',
    'Single_large',
    'Smurfing',
    'Stacked Bipartite',
    'Structuring'
]

#### **Dataset SAML-D e recortes**

##### SAML-D (original)
Dataset original, sem recortes. Geralmente pesado demais para os recursos disponíveis no Google Drive.

> * Nós: 855.460
> * Lavadores: 7.902 (0,92%)
> * Transações: 9.504.852

In [ ]:
# SAML-D original (855.460 nós, 7.902 lavadores, 9.504.852 transações)
saml_d = pd.read_csv('../Datasets/SAML-D/SAML-D.csv')

In [ ]:
df = saml_d
nome_dataset = 'saml_d'

##### SAML-D mini
Primeiros 11% das transações do SAML-D original. Recorte simples feito de maneira bruta, possivelmente com perdas topológicas (estruturas cortadas pela metade)

> * Nós: 325.888
> * Lavadores: 935 (0,29%)
> * Transações: 1.048.575

In [10]:
# SAML-D mini (325.888 nós, 935 lavadores, 1.048.575 transações)
saml_d_mini = pd.read_csv('../Datasets/SAML-D/SAML-D_mini.csv')

In [11]:
df = saml_d_mini
nome_dataset = 'saml_d_mini'

##### SAML-D sem nós exclusivamente remetentes
Foi realizada a remoção iterativa dos nós exclusivamente remetentes, junto com as arestas que neles incidiam, até que não houvesse mais nenhum.

In [ ]:
# SAML-D sem onlysenders (606.260 nós, 6.779 lavadores, 6.678.887 transações)
saml_d_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_nonlysenders.csv')

In [ ]:
df = saml_d_nonlysenders
nome_dataset = 'saml_d_nonlysenders'

##### Metade do SAML-D sem nós exclusivamente remetentes
O algoritmo de remoção de nós exclusivamente remetentes foi aplicado ao SAML-D brutamente cortado pela metade.

In [ ]:
# SAML-D 50% sem onlysenders (443.381 nós, 2.948 lavadores, 2.699.819 transações)
saml_d_50_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_50_nonlysenders.csv')

In [ ]:
df = saml_d_50_nonlysenders
nome_dataset = 'saml_d_50_nonlysenders'

##### Subgrafo Bidirecional do SAML-D

Foram retirados iterativamente nós que não possuíam cardinalidade de entrada, juntamente com suas arestas, até que não houvessem mais nós exclusivamente remetentes. Em seguida, o mesmo foi feito para nós que não possuíam cardinalidade de saída, até que não houvessem mais nós exclusivamente destinatários. O que sobra é o maior subgrafo bidirecional do SAML-D. Ainda há perda de sentido.

>* Nós: 66.370
>* Lavadores: 5.120 (7,71%)
>* Transações: 783.735

In [ ]:
# SAML-D subgrafo bidirecional (66.370 nós, 5.120 lavadores, 783.735 transações)
saml_d_intersection = pd.read_csv('../Datasets/SAML-D/SAML-D_intersection.csv')

In [ ]:
df = saml_d_intersection
nome_dataset = 'saml_d_intersection'

### **Tratamento do dataset para entrada**

#### Conversão das colunas

In [12]:
# Convertendo Date e Time em Timestamp Unix
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time']).astype('int64')

###### Criando mapeamento numérico para labels em string

In [13]:
# Obtendo tipos de agrupamento
currencies          = df.groupby('Payment_currency')['Payment_currency'].count().sort_values(ascending=False).keys()
bank_locations      = df.groupby('Sender_bank_location')['Sender_bank_location'].count().sort_values(ascending=False).keys()
payment_types       = df.groupby('Payment_type')['Payment_type'].count().sort_values(ascending=False).keys()
laundering_types    = df.groupby('Laundering_type')['Laundering_type'].count().sort_values(ascending=False).keys()
accounts            = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
launderers          = pd.concat([
                                  df[df['Is_laundering'] == 1]['Sender_account'],
                                  df[df['Is_laundering'] == 1]['Receiver_account']
                                ]).unique()
is_launderer        = pd.Series(accounts).isin(launderers).astype(int)

# Enumeração dos tipos agrupados
account_map         = {account:i for i, account in enumerate(accounts)}
currency_map        = {currencies[i]:i for i in range(len(currencies))}
bank_location_map   = {bank_locations[i]:i for i in range(len(bank_locations))}
payment_type_map    = {payment_types[i]:i for i in range(len(payment_types))}
laundering_type_map = {laundering_types[i]:i for i in range(len(laundering_types))}

###### Dataset auxiliar de labels

Dataset auxiliar com a classificação de cada conta

In [14]:
# Dataset auxiliar
Accounts = pd.DataFrame({
    'Accounts': accounts,
    'Launderers': pd.Series(accounts).isin(launderers).astype(int)
})

In [15]:
print(f'{'Accounts:':<12}{Accounts.shape[0]:>10}')
print(f'{'Lavadores:':<12}{ launderers.size:>10}')
print(f'{'Transações:':<12}{    df.shape[0]:>10}')

Accounts:       325888
Lavadores:         935
Transações:    1048575


#### **Criação das Features para os nós**

###### Cardinalidade dos nós

In [16]:
#1) Número de transações enviadas
sent_count = df.groupby('Sender_account').size().rename('sent_count')

#2) Número de transações recebidas
received_count = df.groupby('Receiver_account').size().rename('received_count')

###### Valor das transações

Nota: A quantia en cada transação é relativa à moeda de envio.

In [17]:
#3) Volume total já enviado por conta e moeda
sent_amount_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].sum().unstack(fill_value=0)
sent_amount_per_currency.columns = [f'sent_amount_{col.replace(' ', '_').lower()}' for col in sent_amount_per_currency.columns]

#4) Volume total já recebido por conta e moeda
received_amount_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].sum().unstack(fill_value=0)
received_amount_per_currency.columns = [f'received_amount_{col.replace(' ', '_').lower()}' for col in received_amount_per_currency.columns]

In [18]:
# Redundantes? Testar de novo

#5) Valor médio já enviado por conta e moeda
sent_mean_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].mean().unstack(fill_value=0)
sent_mean_per_currency.columns = [f'sent_mean_{col.replace(' ', '_').lower()}' for col in sent_mean_per_currency.columns]

#6) Valor médio já recebido por conta e moeda
received_mean_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].mean().unstack(fill_value=0)
received_mean_per_currency.columns = [f'received_mean_{col.replace(' ', '_').lower()}' for col in received_mean_per_currency.columns]

#7) Maior valor enviado por conta e moeda
sent_max_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].max().unstack(fill_value=0)
sent_max_per_currency.columns = [f'sent_max_{col.replace(' ', '_').lower()}' for col in sent_max_per_currency.columns]

#8) Maior valor recebido por conta e moeda
received_max_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].max().unstack(fill_value=0)
received_max_per_currency.columns = [f'received_max_{col.replace(' ', '_').lower()}' for col in received_max_per_currency.columns]

##### Features avançadas

In [19]:
#9) Percentual de reciprocidade dos vizinhos (para cada aresta de ida, quantas tem volta)

# a) Relações direcionadas distintas: ignora transações repetidas
edges = (
    df[['Sender_account', 'Receiver_account']]
    .drop_duplicates()
    .rename(columns={
        'Sender_account': 'src',
        'Receiver_account': 'dst'
    })
)

# b) Grafo reverso: troca origem e destino
reverse_edges = edges.rename(columns={
    'src': 'dst',
    'dst': 'src'
})

# c) Pares que possuem relações nos dois sentidos
reciprocal_edges = edges.merge(
    reverse_edges,
    on=['src', 'dst'],
    how='inner'
)

# d) Número de vizinhos recíprocos distintos por nó
reciprocal_partners = pd.concat([
    reciprocal_edges[['src', 'dst']].rename(
        columns={'src': 'account', 'dst': 'partner'}
    ),
    reciprocal_edges[['dst', 'src']].rename(
        columns={'dst': 'account', 'src': 'partner'}
    )
]).drop_duplicates()

reciprocal_count = reciprocal_partners.groupby('account').size()

# e) Número de vizinhos distintos de saída e entrada
out_degree = edges.groupby('src').size()
in_degree = edges.groupby('dst').size()

# f) Índice de reciprocidade A (estrutura de vizinhos)
all_accounts = pd.Index(accounts, name='account')

reciprocal_count = reciprocal_count.reindex(
    all_accounts, fill_value=0
)
out_degree = out_degree.reindex(all_accounts, fill_value=0)
in_degree = in_degree.reindex(all_accounts, fill_value=0)

# A interseção é subtraída uma vez para obter a união
union_count = out_degree + in_degree - reciprocal_count

reciprocity_score = (
    reciprocal_count
    .div(union_count.where(union_count > 0))
    .fillna(0.0)
    .rename('reciprocity_score')
)

# g) Índice de reciprocidade B (proporção de vizinhos que enviam de volta)
reciprocity_score_in_out = (
    reciprocal_count
    .div(out_degree)
    .fillna(0.0)
    .rename('reciprocity_score_in_out')
)

# h) Índice de reciprocidade C (proporção de vizinhos que recebem de volta)
reciprocity_score_out_in = (
    reciprocal_count
    .div(in_degree)
    .fillna(0.0)
    .rename('reciprocity_score_out_in')
)

In [20]:
#10) Histograma dos tipos de pagamento utilizados pelo nó
payment_type_sender_counts = df.groupby(['Sender_account', 'Payment_type']).size().unstack(fill_value=0)
payment_type_sender_counts.columns = [f'sent_payment_type_{col.replace(' ', '_').lower()}' for col in payment_type_sender_counts.columns]


In [21]:
"""#11) Total de moedas já utilizadas pelo nó
sender_currencies = df.groupby('Sender_account')['Payment_currency'].apply(lambda x: set(x))
receiver_currencies = df.groupby('Receiver_account')['Received_currency'].apply(lambda x: set(x))
unique_currencies_per_account = {}
for acc in accounts:
    sent_c = sender_currencies.get(acc, set())
    rec_c = receiver_currencies.get(acc, set())
    unique_currencies_per_account[acc] = sent_c.union(rec_c)
currency_diversity = pd.Series({acc: len(currencies) for acc, currencies in unique_currencies_per_account.items()}, name='currency_diversity')
"""

"#11) Total de moedas já utilizadas pelo nó\nsender_currencies = df.groupby('Sender_account')['Payment_currency'].apply(lambda x: set(x))\nreceiver_currencies = df.groupby('Receiver_account')['Received_currency'].apply(lambda x: set(x))\nunique_currencies_per_account = {}\nfor acc in accounts:\n    sent_c = sender_currencies.get(acc, set())\n    rec_c = receiver_currencies.get(acc, set())\n    unique_currencies_per_account[acc] = sent_c.union(rec_c)\ncurrency_diversity = pd.Series({acc: len(currencies) for acc, currencies in unique_currencies_per_account.items()}, name='currency_diversity')\n"

In [22]:
#12) Frequência de cada moeda utilizada pelo nó
currency_sender_counts = df.groupby(['Sender_account', 'Payment_currency']).size().unstack(fill_value=0)
currency_sender_counts.columns = [f'sent_currency_{col.replace(' ', '_').lower()}' for col in currency_sender_counts.columns]
currency_receiver_counts = df.groupby(['Receiver_account', 'Received_currency']).size().unstack(fill_value=0)
currency_receiver_counts.columns = [f'received_currency_{col.replace(' ', '_').lower()}' for col in currency_receiver_counts.columns]
all_currency_counts_df = pd.DataFrame(index=accounts)
all_currency_counts_df = all_currency_counts_df.merge(currency_sender_counts, left_index=True, right_index=True, how='left').fillna(0)
all_currency_counts_df = all_currency_counts_df.merge(currency_receiver_counts, left_index=True, right_index=True, how='left').fillna(0)


##### Adição das features criadas em Accounts

In [23]:
# Adicionando features        Coluna adicionada:
Accounts = Accounts.merge(    sent_count                      , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    received_count                  , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    sent_amount_per_currency        , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    received_amount_per_currency    , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    sent_mean_per_currency          , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    received_mean_per_currency      , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    sent_max_per_currency           , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    received_max_per_currency       , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    reciprocity_score               , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    reciprocity_score_in_out        , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    reciprocity_score_out_in        , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    payment_type_sender_counts      , left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(    all_currency_counts_df          , left_on='Accounts', right_index=True, how='left').fillna(0)

# Identificando features para normalização
cols_to_normalize = [col for col in Accounts.columns[2:]]

# Aplicando StandardScaler
scaler = StandardScaler()
Accounts[cols_to_normalize] = scaler.fit_transform(Accounts[cols_to_normalize])

#### **Preparando variáveis de entrada**

##### Matriz de adjacência (COO Matrix)

###### Matriz original (adj_original)

In [24]:
# Convertendo dataframe em matriz esparsa

row = df['Sender_account'].map(account_map)
col = df['Receiver_account'].map(account_map)

values = np.ones(df.shape[0])

num_nodes = Accounts.shape[0]

adj_original = coo_matrix((values, (row, col)), shape=(num_nodes, num_nodes))
adj = adj_original

###### Matriz bidirecional (adj_bi)
Transforma o grafo original em um grafo bidirecional

In [25]:
# Transformação do grafo para grafo bidirecional
adj_bi = adj + adj.T
adj_bi.data = np.ones_like(adj_bi.data)

###### Matriz do grafo manualmente conexo (adj_conexo)

Foram formadas pequenas pontes sobre as diversas "ilhas" desconexas de nós para garantir message passing sobre todo o grafo

In [26]:
adj = adj_original
# Obtendo componentes conexas
n_components, labels_cc = connected_components(adj)
print('Número de componentes em adj_original:', n_components )

# Lista de componentes (os nós que compõem a componente i)
components = [[] for _ in range(n_components)]
for node_index, component_label in enumerate(labels_cc):
    components[component_label].append(node_index)

adj_lil = adj.tolil() # Formato que permite manipulação do coo_sparse

# Conectando (bidirecionalmente) componentes (peso desbalanceado, estudar)
for i in range(-1, n_components-1):
  adj_lil[ components[i][0], components[i+1][0] ] = 1.0
  adj_lil[ components[i+1][0], components[i][0] ] = 1.0

adj_conexo = adj_lil.tocoo()
print('Número de componentes em adj_conexo:', connected_components(adj_conexo)[0] )

Número de componentes em adj_original: 20250
Número de componentes em adj_conexo: 1


In [27]:
adj = adj_bi
# Obtendo componentes conexas
n_components, labels_cc = connected_components(adj)
print('Número de componentes em adj_bi:', n_components )

# Lista de componentes (os nós que compõem a componente i)
components = [[] for _ in range(n_components)]
for node_index, component_label in enumerate(labels_cc):
    components[component_label].append(node_index)

adj_lil = adj.tolil() # Formato que permite manipulação do coo_sparse

# Conectando (bidirecionalmente) componentes (peso desbalanceado, estudar)
for i in range(-1, n_components-1):
  adj_lil[ components[i][0], components[i+1][0] ] = 1.0
  adj_lil[ components[i+1][0], components[i][0] ] = 1.0

adj_bi_conexo = adj_lil.tocoo()
print('Número de componentes em adj_bi_conexo:', connected_components(adj_bi_conexo)[0] )

Número de componentes em adj_bi: 20250
Número de componentes em adj_bi_conexo: 1


##### Vetor de Features

In [28]:
# Features dos nós
feats = Accounts.drop(columns=['Accounts', 'Launderers']).values

##### Vetor de Labels

In [29]:
# Labels dos nós (fraudadores ou não-fraudadores)
labels = np.array(is_launderer)

##### Vetor de Mascaramento

In [30]:
from logging import error
# Configurando mascaramento dos nós

train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

# Divisórias percentuais de mascaramento
percent_trn = 0.4
percent_val = 0.3
percent_tst = 0.3
if percent_trn+percent_val+percent_tst != 1.0:
  error('Atenção: Divisão do mascaramento incorreta! A soma dos intervalos percentuais escolhidos não está dando 100%.')

# Criando fatias
trn_slice = slice(None, int(num_nodes*percent_trn))
val_slice = slice(int(num_nodes*percent_trn), -int(num_nodes*percent_tst))
tst_slice = slice(-int(num_nodes*percent_tst), None)

# Mascaramento:
train_mask[trn_slice] = True
val_mask[val_slice] = True
test_mask[tst_slice] = True


#### **Gravando variáveis de entrada**

In [31]:
# Cria pasta para armazenar as variáveis, caso não exista
if not os.path.exists(f'datasets/{nome_dataset}'):
    os.makedirs(f'datasets/{nome_dataset}')

##### Matriz de adjacência

In [32]:
# Grafo original
pickle.dump(adj_original, open(f'datasets/{nome_dataset}/adj_original.pkl', 'wb'))

In [33]:
# Grafo bidirecionado
pickle.dump(adj_bi, open(f'datasets/{nome_dataset}/adj_bi.pkl', 'wb'))

In [34]:
# Grafo conexo gerado diretamente a partir do original
pickle.dump(adj_conexo, open(f'datasets/{nome_dataset}/adj_conexo.pkl', 'wb'))

In [35]:
# Grafo conexo gerado a partir do grafo  bidirecionado
pickle.dump(adj_bi_conexo, open(f'datasets/{nome_dataset}/adj_bi_conexo.pkl', 'wb'))

##### Vetor de Features

In [36]:
pickle.dump(feats, open(f'datasets/{nome_dataset}/feats.pkl', 'wb'))

##### Vetor de Labels

In [37]:
pickle.dump(labels, open(f'datasets/{nome_dataset}/label.pkl', 'wb'))

##### Vetor de Mascaramento

In [38]:
nome_mask = f'{int(percent_trn*100)}_{int(percent_val*100)}_{int(percent_tst*100)}'
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open(f'datasets/{nome_dataset}/mask_{nome_mask}.pkl', 'wb'))

# **Execução do OpenGraph**

##### Rodando o modelo

In [8]:
nome_tstdata = [
    'saml_d',
    'saml_d_mini',
    'saml_d_nonlysenders',
    'saml_d_50_nonlysenders',
    'saml_d_intersection'
    ]
nome_adj_file = [
    'original',
    'bi',
    'conexo',
    'bi_conexo'
    ]
nome_mask_file = ['40_30_30']

In [3]:
%cd node_classification

/dados/luiz.gontijo/TCC/OpenGraph/node_classification


In [8]:
for tstdata in nome_tstdata:
    for adj_file in nome_adj_file:
        !python main.py --epoch 0 --tstdata {tstdata} --adj_file {adj_file} --mask_file 40_30_30

2026-11-10 00:13:40.877540: Start
Dataset: saml_d_mini, Node num: 325888, Edge num: 313179
/dados/luiz.gontijo/TCC/OpenGraph/node_classification/data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])
/dados/luiz.gontijo/TCC/OpenGraph/node_classification/data_handler.py:115: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  asym_adj = t.sparse.FloatTensor(idxs, vals, shape)
2026-11-10 00:13:43.307122: Load Data
Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0
2026-11-10 00:13:43.432656: Model Prepared
2026-11-10 00:13:44.598276: Epoch 0/0, saml_d_mini Test: Acc = 0.9993, F1 = 0.4998      
2026-11-10 00:13:48.343685: Epoch 0/0, saml_d_mini AVG: Acc = 0.9993, F1 = 0.4998      
2026-11-10 00:13:50.438211:

In [9]:
%cd ..

/dados/luiz.gontijo/TCC/OpenGraph


##### Resultado

In [11]:
for tstdata in nome_tstdata:
    for adj_file in nome_adj_file:
      for mask_file in ['40_30_30']:
        !echo {tstdata} {adj_file} {mask_file}
        !ls node_classification/Resultados/predicoes/{tstdata}/adj_{adj_file}_mask_{mask_file}
        !ls node_classification/Resultados/embeddings/{tstdata}/adj_{adj_file}_mask_{mask_file}
        !echo ''

saml_d original 40_30_30
predict0.pkl


embedding0.pkl

saml_d bi 40_30_30
predict0.pkl
embedding0.pkl

saml_d conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d bi_conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_mini original 40_30_30
predict0.pkl
embedding0.pkl

saml_d_mini bi 40_30_30
predict0.pkl
embedding0.pkl

saml_d_mini conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_mini bi_conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_nonlysenders original 40_30_30
predict0.pkl
embedding0.pkl

saml_d_nonlysenders bi 40_30_30
predict0.pkl
embedding0.pkl

saml_d_nonlysenders conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_nonlysenders bi_conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_50_nonlysenders original 40_30_30
predict0.pkl
embedding0.pkl

saml_d_50_nonlysenders bi 40_30_30
predict0.pkl
embedding0.pkl

saml_d_50_nonlysenders conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_50_nonlysenders bi_conexo 40_30_30
predict0.pkl
embedding0.pkl

saml_d_intersection original 40_30_30
predict0.pkl
embedding0.pkl


In [17]:
for tstdata in nome_tstdata:
    for adj_file in nome_adj_file:
      for mask_file in nome_mask_file:
        bizoia_resultados('OpenGraph', 1, tstdata, adj_file, mask_file, 325888)

--- Resultado 0 ---


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

# Análise dos Resultados

## Cálculo e registro de métricas

In [9]:
resultados_df = pd.DataFrame(columns=['modelo', 'dataset', 'adj_file', 'mask', 'qnt_zeroes', 'qnt_ones', 'accuracy', 'precision', 'recall', 'f1_score'])

# Loops para automatizar a gravação dos resultados
for tstdata in nome_tstdata:
    for adj_file in nome_adj_file:
        for mask_file in nome_mask_file:

            ######## OpenGraph ########

            # Obtendo o mascaramento utilizado
            with open(f'datasets/{tstdata}/mask_{mask_file}.pkl', 'rb') as f:
                mask = pickle.load(f)

            # Obtendo as labels do grafo
            with open(f'datasets/{tstdata}/label.pkl', 'rb') as f:
                labels = pickle.load(f)

            # Transformando resultados das predições em DataFrame
            with open(f'node_classification/Resultados/predicoes/{tstdata}/adj_{adj_file}_mask_{mask_file}/predict0.pkl', 'rb') as f:
                data = pickle.load(f)
            resultado = pd.DataFrame.from_dict({'preds': data['preds'], 'labels': data['labels']})

            # Transformando o embedding gerado em DataFrame
            with open(f'node_classification/Resultados/embeddings/{tstdata}/adj_{adj_file}_mask_{mask_file}/embedding0.pkl', 'rb') as f:
                embed = pickle.load(f)
            embed_df = pd.DataFrame(torch.cat([e.cpu() for e in embed], dim=0).numpy())

            ##### Outros modelos ######


            # Divisão de treinamento alinhada com o mascaramento
            train_x = embed_df.iloc[~mask['test']]
            test_x = embed_df.iloc[mask['test']]
            train_y = labels[~mask['test']]
            test_y = labels[mask['test']]

            # Predição do RandomForest
            classificador_randomforest = RandomForestClassifier(n_estimators=200, random_state=18, max_depth=4, class_weight='balanced')
            classificador_randomforest.fit(train_x, train_y)
            pred_RandomForest = classificador_randomforest.predict(test_x)

            # Predição do XGBoost
            scale_pos_weight = (train_y == 0).sum() / (train_y == 1).sum()
            classificador_xgb = xgb.XGBClassifier(objective='binary:logistic', random_state=18, scale_pos_weight=scale_pos_weight)
            classificador_xgb.fit(train_x, train_y)
            pred_XGBoost = classificador_xgb.predict(test_x)

            #### Calculando e registrando métricas ####
            for modelo, test_y, pred_y in [('OpenGraph', resultado['labels'], resultado['preds']), ('Random Forest', test_y, pred_RandomForest), ('XGBoost', test_y, pred_XGBoost)]:
                qnt_zeroes, qnt_ones, accuracy, precision, recall, f1 = calcula_metricas(test_y, pred_y)
                resultados_df = pd.concat([resultados_df, pd.DataFrame({
                    'modelo': [modelo],
                    'dataset': [tstdata],
                    'adj_file': [adj_file],
                    'mask': [mask_file],
                    'qnt_zeroes': [qnt_zeroes],
                    'qnt_ones': [qnt_ones],
                    'accuracy': [accuracy],
                    'precision': [precision],
                    'recall': [recall],
                    'f1_score': [f1]
                })], ignore_index=True)






In [10]:
!ls

'Ações do GitHub.ipynb'		 Models		    imgs
'Análise dos Resultados.ipynb'	 OpenGraph.ipynb    link_prediction
'Análise dos Resultados.ipynb'	 README.md	    node_classification
 History			 datasets	    requirements.txt
 LICENSE			 graph_generation   resultados_df.csv


In [11]:
## Salvando resultados em CSV
resultados_df.to_csv('resultados_df.csv', index=False)

In [12]:
resultados_df

,modelo,dataset,adj_file,mask,qnt_zeroes,qnt_ones,accuracy,precision,recall,f1_score
0,OpenGraph,saml_d,original,40_30_30,256638,0,0.997576,0.0,0.0,0.0
1,Random Forest,saml_d,original,40_30_30,256638,0,0.997576,0.0,0.0,0.0
2,XGBoost,saml_d,original,40_30_30,256638,0,0.997576,0.0,0.0,0.0
3,OpenGraph,saml_d,bi,40_30_30,103503,153135,0.403319,0.002044,0.503215,0.004071
4,Random Forest,saml_d,bi,40_30_30,204133,52505,0.794664,0.004095,0.345659,0.008094
5,XGBoost,saml_d,bi,40_30_30,237700,18938,0.924392,0.004119,0.125402,0.007975
6,OpenGraph,saml_d,conexo,40_30_30,256638,0,0.997576,0.0,0.0,0.0
7,Random Forest,saml_d,conexo,40_30_30,256638,0,0.997576,0.0,0.0,0.0
8,XGBoost,saml_d,conexo,40_30_30,256638,0,0.997576,0.0,0.0,0.0
9,OpenGraph,saml_d,bi_conexo,40_30_30,162830,93808,0.634177,0.00291,0.438907,0.005782
